# nb14: Effective Mass + Janus Asymmetry Descriptors

Two new descriptor families:
1. **Effective mass at VBM/CBM** from vasprun.xml band structure (parabolic fit)
2. **Janus / out-of-plane structural asymmetry** from POSCAR_std (z-dipoles, layer composition)

Both are exact (not proxies). Both target Rashba physics directly.


In [ ]:
import os, glob, warnings, numpy as np, pandas as pd
from time import time
from itertools import combinations
from urllib.parse import unquote

from pymatgen.core import Structure
from pymatgen.io.vasp.outputs import Vasprun

from sklearn.model_selection import LeaveOneOut, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

warnings.filterwarnings('ignore')

BASE_DIR = os.path.abspath(os.path.join('..'))
OLD_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors_old.csv')
NEW_CSV = os.path.join(BASE_DIR, 'k-path', 'old+new_nb6', 'rashba_206_all_descriptors.csv')
POSCAR_DIR = os.path.join(BASE_DIR, 'Inverse-design', 'rashba')
RESULTS_DIR = os.path.join('.', 'nb14_effmass_janus-results')
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Setup OK. Results dir:', RESULTS_DIR)


## Cell 2: Load merged data, baseline (sanity check)

In [ ]:
df_old = pd.read_csv(OLD_CSV)
df_new = pd.read_csv(NEW_CSV)
ID_COLS = ['Formula', 'uid', 'kpath', 'Rashba_parameter']
TARGET = 'Rashba_parameter'

df_merged = df_old[ID_COLS].copy()
old_features = [c for c in df_old.columns if c not in ID_COLS]
new_features = [c for c in df_new.columns if c not in ID_COLS]
overlap = set(old_features) & set(new_features)
for col in old_features:
    df_merged[f'old_{col}' if col in overlap else col] = df_old[col].values
for col in new_features:
    df_merged[f'new_{col}' if col in overlap else col] = df_new[col].values

idx_max = df_merged.groupby('uid')[TARGET].idxmax()
df_99 = df_merged.loc[idx_max].reset_index(drop=True)
y_99 = df_99[TARGET].values

XGB_REG_PARAMS = dict(n_estimators=100, max_depth=3, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, reg_alpha=1.0, reg_lambda=1.0,
    random_state=42, verbosity=0)
BASELINE_RAW = ['E_pfrac_VBM', 'E_pfrac_CBM', 'radius_mean',
                'pmid_afs_gauss_std', 'kpath_angle_deg', 'ehull']

def resolve(name, cols):
    if name in cols: return name
    if f'old_{name}' in cols: return f'old_{name}'
    if f'new_{name}' in cols: return f'new_{name}'
    raise KeyError(name)

BASELINE = [resolve(n, df_99.columns) for n in BASELINE_RAW]

def eval_reg_99(features, df, y, seed=42):
    X = df[features].fillna(0).values
    params = dict(XGB_REG_PARAMS); params['random_state'] = seed
    model = XGBRegressor(**params)
    y_pred = cross_val_predict(model, X, y, cv=LeaveOneOut())
    return r2_score(y, y_pred), mean_absolute_error(y, y_pred)

r2_base, mae_base = eval_reg_99(BASELINE, df_99, y_99)
print(f'Baseline (C6) LOO R2 = {r2_base:.4f}, MAE = {mae_base:.4f}')
if abs(r2_base - 0.643) > 0.05:
    print('  WARNING baseline differs from nb7 by > 0.05')


## Cell 3: Janus / out-of-plane asymmetry features (POSCAR_std)

For each compound, compute geometric quantities that capture inversion-symmetry breaking perpendicular to the 2D layer.

**Why z-dipole matters:** in 2D Rashba, the field gradient that drives SOC splitting comes from charge redistribution perpendicular to the layer. If atoms are arranged symmetrically about the midplane in z, no field. If heavy atoms are biased toward one face (Janus structure), large field → strong Rashba.

**z-dipole weighted by Z:** signed sum of (z_i − z_mean) × Z_i, normalized. Zero if heavy atoms are symmetric about the midplane. Non-zero if biased to one side.

In [ ]:
def load_structure(uid):
    matches = glob.glob(os.path.join(POSCAR_DIR, f'*-{uid}', 'POSCAR_std'))
    if not matches:
        return None
    try:
        return Structure.from_file(matches[0])
    except Exception:
        return None


JANUS_KEYS = [
    'janus_z_range', 'janus_z_std', 'janus_n_layers',
    'janus_z_dipole_Z', 'janus_z_dipole_X', 'janus_z_dipole_mass',
    'janus_z_dipole_Z_abs', 'janus_z_dipole_X_abs', 'janus_z_dipole_mass_abs',
    'janus_top_bot_meanZ', 'janus_top_bot_meanX',
    'janus_top_bot_meanZ_abs', 'janus_top_bot_meanX_abs',
]


def janus_features(struct):
    if struct is None:
        return {k: np.nan for k in JANUS_KEYS}

    zs = struct.cart_coords[:, 2]
    z_center = zs.mean()
    z_centered = zs - z_center

    Zs = np.array([s.specie.Z for s in struct], dtype=float)
    Xs = np.array([float(s.specie.X) if s.specie.X is not None else 0.0
                   for s in struct], dtype=float)
    masses = np.array([float(s.specie.atomic_mass) for s in struct], dtype=float)

    feats = {}
    feats['janus_z_range'] = float(zs.max() - zs.min())
    feats['janus_z_std'] = float(zs.std())

    # Number of distinct z-layers (cluster within 0.5 A)
    sorted_z = np.sort(zs)
    n_layers = 1
    for i in range(1, len(sorted_z)):
        if sorted_z[i] - sorted_z[i - 1] > 0.5:
            n_layers += 1
    feats['janus_n_layers'] = n_layers

    # Weighted z-dipoles: signed quantities
    feats['janus_z_dipole_Z'] = float((z_centered * Zs).sum() / max(Zs.sum(), 1e-9))
    feats['janus_z_dipole_X'] = float((z_centered * Xs).sum() / max(Xs.sum(), 1e-9))
    feats['janus_z_dipole_mass'] = float((z_centered * masses).sum() / max(masses.sum(), 1e-9))

    # Magnitudes (the model can use either sign or magnitude)
    feats['janus_z_dipole_Z_abs'] = abs(feats['janus_z_dipole_Z'])
    feats['janus_z_dipole_X_abs'] = abs(feats['janus_z_dipole_X'])
    feats['janus_z_dipole_mass_abs'] = abs(feats['janus_z_dipole_mass'])

    # Top vs bottom layer composition difference
    top_mask = zs > z_center
    bot_mask = ~top_mask
    if top_mask.any() and bot_mask.any():
        feats['janus_top_bot_meanZ'] = float(Zs[top_mask].mean() - Zs[bot_mask].mean())
        feats['janus_top_bot_meanX'] = float(Xs[top_mask].mean() - Xs[bot_mask].mean())
    else:
        feats['janus_top_bot_meanZ'] = 0.0
        feats['janus_top_bot_meanX'] = 0.0
    feats['janus_top_bot_meanZ_abs'] = abs(feats['janus_top_bot_meanZ'])
    feats['janus_top_bot_meanX_abs'] = abs(feats['janus_top_bot_meanX'])
    return feats


print('Computing Janus features for 99 compounds...')
janus_rows = []
n_failed = 0
t0 = time()
for i, row in df_99.iterrows():
    struct = load_structure(row['uid'])
    if struct is None:
        n_failed += 1
    feats = janus_features(struct)
    feats['uid'] = row['uid']
    janus_rows.append(feats)

janus_df = pd.DataFrame(janus_rows)
print(f'Done in {time()-t0:.1f}s. Failed: {n_failed}/99. Features: {len(JANUS_KEYS)}')
print('\nSample (first 5 rows):')
print(janus_df.head())

# Quick sanity check: BiITe polymorphs - do they differ in Janus features?
print('\nBiITe polymorphs check (the polymorph case from nb12):')
biite_rows = df_99[df_99['Formula'] == 'BiITe']
for _, row in biite_rows.iterrows():
    j = janus_df[janus_df['uid'] == row['uid']].iloc[0]
    print(f"  uid={row['uid']}, alpha_R={row[TARGET]:.3f}, "
          f"z_dipole_Z={j['janus_z_dipole_Z']:+.4f}, top_bot_meanZ={j['janus_top_bot_meanZ']:+.3f}")

janus_df.to_csv(os.path.join(RESULTS_DIR, 'janus_features.csv'), index=False)


## Cell 4: Effective mass at VBM and CBM (vasprun.xml parsing)

For each compound, parse vasprun.xml to get the SOC band structure, find VBM and CBM, fit a parabola in a small window around each band edge along the k-path, and extract effective mass m\*.

**Conversion:** in atomic units, 1/m\* = (1/ℏ²) d²E/dk². Numerically, fitting E(k) = a + bk + ck² gives m\* = (ℏ²/2c). With E in eV and k in 1/Å, the prefactor is ℏ²/(2 m_e) = 3.81 eV·Å² so m\* (in m_e) = 3.81/(2c).

**Time estimate:** ~2 sec per compound for parsing vasprun.xml ≈ 3-5 min total for 99 compounds.

In [ ]:
def find_vasprun(uid):
    folder_glob = glob.glob(os.path.join(POSCAR_DIR, f'*-{uid}'))
    if not folder_glob:
        return None
    folder = folder_glob[0]
    for f in os.listdir(folder):
        if unquote(f).endswith('/vasprun.xml'):
            return os.path.join(folder, f)
    return None


BS_KEYS = [
    'bs_m_eff_VBM', 'bs_m_eff_CBM', 'bs_m_eff_ratio',
    'bs_VBM_kdist_to_gamma', 'bs_CBM_kdist_to_gamma',
    'bs_VBM_at_gamma', 'bs_CBM_at_gamma',
    'bs_band_gap_at_VBM_kpt', 'bs_band_gap_at_CBM_kpt',
    'bs_VBM_curvature', 'bs_CBM_curvature',
]


def parabolic_fit_curvature(eigvals_band, kpoints_cart, kidx, window=3):
    """Fit parabola E(d) = a + b*d + c*d^2 in window around kidx along path.
    d = signed distance from kidx along path direction (1/A).
    Returns (m_star_in_me_units, c)."""
    n = eigvals_band.shape[0]
    i_lo = max(0, kidx - window)
    i_hi = min(n, kidx + window + 1)
    if i_hi - i_lo < 3:
        return np.nan, np.nan

    k0 = kpoints_cart[kidx]
    ds = np.array([np.linalg.norm(kpoints_cart[j] - k0) for j in range(i_lo, i_hi)])
    # signed distance: negative for j < kidx
    for j in range(i_lo, i_hi):
        if j < kidx:
            ds[j - i_lo] = -ds[j - i_lo]
    Es = np.array([eigvals_band[j] for j in range(i_lo, i_hi)])

    try:
        coef = np.polyfit(ds, Es, 2)
        c = float(coef[0])
        if abs(c) < 1e-9:
            return np.nan, c
        # m* in m_e units: 3.81 eV*A^2 = hbar^2/(2 m_e)
        m_star = 3.81 / (2.0 * c)
        return float(m_star), c
    except Exception:
        return np.nan, np.nan


def effmass_features(uid):
    vr_path = find_vasprun(uid)
    if vr_path is None:
        return {k: np.nan for k in BS_KEYS}, 'no vasprun.xml'

    try:
        vr = Vasprun(vr_path, parse_dos=False, parse_potcar_file=False,
                     exception_on_bad_xml=False)
        bs = vr.get_band_structure()
    except Exception as e:
        return {k: np.nan for k in BS_KEYS}, f'parse failed: {e}'

    try:
        # Eigenvalues: dict spin -> [n_bands, n_kpoints]
        eigvals_dict = bs.bands
        eigvals = list(eigvals_dict.values())[0]   # [n_bands, n_kpoints]

        # k-points
        kpoints_frac = np.array([kp.frac_coords for kp in bs.kpoints])
        rec = bs.lattice_rec.matrix   # 3x3, in 1/A; rows are b1, b2, b3
        kpoints_cart = kpoints_frac @ rec   # [n_kpoints, 3]

        # VBM, CBM from pymatgen
        vbm = bs.get_vbm()
        cbm = bs.get_cbm()
        if not vbm or not cbm:
            return {k: np.nan for k in BS_KEYS}, 'no vbm/cbm'

        # Resolve band/kpoint indices (pymatgen returns various shapes)
        def _resolve_idx(d, key):
            v = d.get(key)
            if v is None:
                return None
            if isinstance(v, dict):
                v = list(v.values())[0]
            if isinstance(v, (list, tuple)):
                v = v[0] if len(v) > 0 else None
            return int(v) if v is not None else None

        vbm_band = _resolve_idx(vbm, 'band_index')
        vbm_kidx = _resolve_idx(vbm, 'kpoint_index')
        cbm_band = _resolve_idx(cbm, 'band_index')
        cbm_kidx = _resolve_idx(cbm, 'kpoint_index')

        if any(x is None for x in [vbm_band, vbm_kidx, cbm_band, cbm_kidx]):
            return {k: np.nan for k in BS_KEYS}, 'index resolution failed'

        # k-distances to gamma (gamma is at fractional 0,0,0)
        vbm_kdist = float(np.linalg.norm(kpoints_cart[vbm_kidx]))
        cbm_kdist = float(np.linalg.norm(kpoints_cart[cbm_kidx]))

        # Effective mass
        mstar_vbm, curv_vbm = parabolic_fit_curvature(eigvals[vbm_band], kpoints_cart, vbm_kidx)
        mstar_cbm, curv_cbm = parabolic_fit_curvature(eigvals[cbm_band], kpoints_cart, cbm_kidx)

        # Ratio (use absolute VBM mass since it's negative for max)
        if not np.isnan(mstar_vbm) and not np.isnan(mstar_cbm) and abs(mstar_vbm) > 1e-9:
            m_ratio = float(mstar_cbm / abs(mstar_vbm))
        else:
            m_ratio = np.nan

        # Vertical gap at VBM and CBM k-points
        gap_at_vbm_k = float(eigvals[cbm_band, vbm_kidx] - eigvals[vbm_band, vbm_kidx])
        gap_at_cbm_k = float(eigvals[cbm_band, cbm_kidx] - eigvals[vbm_band, cbm_kidx])

        return {
            'bs_m_eff_VBM': mstar_vbm,
            'bs_m_eff_CBM': mstar_cbm,
            'bs_m_eff_ratio': m_ratio,
            'bs_VBM_kdist_to_gamma': vbm_kdist,
            'bs_CBM_kdist_to_gamma': cbm_kdist,
            'bs_VBM_at_gamma': 1 if vbm_kdist < 0.05 else 0,
            'bs_CBM_at_gamma': 1 if cbm_kdist < 0.05 else 0,
            'bs_band_gap_at_VBM_kpt': gap_at_vbm_k,
            'bs_band_gap_at_CBM_kpt': gap_at_cbm_k,
            'bs_VBM_curvature': curv_vbm if not np.isnan(curv_vbm) else np.nan,
            'bs_CBM_curvature': curv_cbm if not np.isnan(curv_cbm) else np.nan,
        }, 'OK'
    except Exception as e:
        return {k: np.nan for k in BS_KEYS}, f'extraction failed: {e}'


print('Extracting effective mass + band edge features for 99 compounds...')
print('  (parsing vasprun.xml takes ~2 sec per compound)')
bs_rows = []
status_counter = {}
t0 = time()
for i, row in df_99.iterrows():
    feats, status = effmass_features(row['uid'])
    feats['uid'] = row['uid']
    feats['_status'] = status
    bs_rows.append(feats)
    status_counter[status] = status_counter.get(status, 0) + 1
    if (i + 1) % 20 == 0:
        print(f'  [{i+1}/99] elapsed {time()-t0:.0f}s')

bs_df = pd.DataFrame(bs_rows)
print(f'\nDone in {time()-t0:.0f}s.')
print(f'Status counts:')
for s, n in status_counter.items():
    print(f'  {s}: {n}')

# Drop status col before saving features
bs_features_df = bs_df.drop(columns=['_status'])
print(f'\nSample:')
print(bs_features_df.head())

# Sanity check: BiITe polymorphs again
print('\nBiITe polymorphs check (band-structure features):')
biite_rows = df_99[df_99['Formula'] == 'BiITe']
for _, row in biite_rows.iterrows():
    b = bs_df[bs_df['uid'] == row['uid']].iloc[0]
    print(f"  uid={row['uid']}, alpha_R={row[TARGET]:.3f}, "
          f"m_eff_VBM={b['bs_m_eff_VBM']:+.3f}, m_eff_CBM={b['bs_m_eff_CBM']:+.3f}, "
          f"VBM_at_gamma={b['bs_VBM_at_gamma']}")

bs_features_df.to_csv(os.path.join(RESULTS_DIR, 'effmass_features.csv'), index=False)


## Cell 5: Combine features with df_99

In [ ]:
# Merge both feature dataframes onto df_99 by uid
df_99_ext = df_99.merge(janus_df, on='uid', how='left').merge(bs_features_df, on='uid', how='left')
print(f'Extended df_99: {df_99_ext.shape[0]} rows, {df_99_ext.shape[1]} cols')

# All new candidates from nb14
NEW_CANDIDATES = JANUS_KEYS + BS_KEYS

# Drop constant columns and all-NaN
nuniq = df_99_ext[NEW_CANDIDATES].nunique(dropna=True)
NEW_CANDIDATES = [c for c in NEW_CANDIDATES if nuniq.get(c, 0) > 1]
print(f'Active new candidates after dropping constants/all-NaN: {len(NEW_CANDIDATES)}')
for c in NEW_CANDIDATES:
    n_nan = df_99_ext[c].isna().sum()
    print(f'  {c:35s}  NaN: {n_nan}/99')


## Cell 6: Phase A — C6 + 1

In [ ]:
print('=' * 70)
print('  PHASE A: C6 + 1 (one new feature at a time)')
print('=' * 70)

phase_a = []
t0 = time()
for cand in NEW_CANDIDATES:
    feats = BASELINE + [cand]
    try:
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        phase_a.append({'candidate': cand, 'r2': r2, 'mae': mae,
                        'delta_r2': r2 - r2_base})
    except Exception as e:
        print(f'  {cand}: FAILED -- {e}')

phase_a_df = pd.DataFrame(phase_a).sort_values('delta_r2', ascending=False).reset_index(drop=True)
print(f'\nDone in {time()-t0:.0f}s.')
print(f'Baseline: R2 = {r2_base:.4f}\n')
print('All C6 + 1 candidates (sorted by delta_r2):')
print(phase_a_df.to_string(index=False))
phase_a_df.to_csv(os.path.join(RESULTS_DIR, 'phase_a.csv'), index=False)


## Cell 7: Phase B — C6 + 2 from top features

Take the top 5 single-feature winners. Test all C(5,2) = 10 pairs.

In [ ]:
K = min(5, len(phase_a_df))
top_k = phase_a_df.head(K)['candidate'].tolist()
print(f'Top {K} from Phase A: {top_k}\n')

phase_b = []
for c1, c2 in combinations(top_k, 2):
    feats = BASELINE + [c1, c2]
    r2, mae = eval_reg_99(feats, df_99_ext, y_99)
    phase_b.append({'cand_1': c1, 'cand_2': c2, 'r2': r2, 'mae': mae,
                    'delta_r2': r2 - r2_base})

phase_b_df = pd.DataFrame(phase_b).sort_values('delta_r2', ascending=False).reset_index(drop=True)
print('All C6 + 2 pairs:')
print(phase_b_df.to_string(index=False))
phase_b_df.to_csv(os.path.join(RESULTS_DIR, 'phase_b.csv'), index=False)

best_a = phase_a_df.iloc[0]['r2']
best_b = phase_b_df.iloc[0]['r2']
print(f'\nBaseline:  R2 = {r2_base:.4f}')
print(f'Best A:    R2 = {best_a:.4f}  ({best_a - r2_base:+.4f})')
print(f'Best B:    R2 = {best_b:.4f}  ({best_b - r2_base:+.4f})')


## Cell 8: Phase C — C6 + 3 (only if Phase B improved over Phase A)

In [ ]:
if phase_b_df.iloc[0]['r2'] > phase_a_df.iloc[0]['r2']:
    phase_c = []
    for c1, c2, c3 in combinations(top_k, 3):
        feats = BASELINE + [c1, c2, c3]
        r2, mae = eval_reg_99(feats, df_99_ext, y_99)
        phase_c.append({'cand_1': c1, 'cand_2': c2, 'cand_3': c3,
                        'r2': r2, 'mae': mae, 'delta_r2': r2 - r2_base})
    phase_c_df = pd.DataFrame(phase_c).sort_values('delta_r2', ascending=False).reset_index(drop=True)
    print('Phase C (C6 + 3):')
    print(phase_c_df.head(10).to_string(index=False))
    phase_c_df.to_csv(os.path.join(RESULTS_DIR, 'phase_c.csv'), index=False)
    best_c = phase_c_df.iloc[0]['r2']
    print(f'\nBest C: R2 = {best_c:.4f}')
else:
    print('Phase B did not improve over A. Skipping Phase C.')
    phase_c_df = pd.DataFrame()


## Cell 9: Multi-seed validation on the best feature set

If we found something promising (delta_r2 > 0.02), validate by running 10 seeds. If single-seed gain is real, multi-seed mean stays high; if it's noise, multi-seed mean collapses toward baseline.

In [ ]:
# Pick the best feature set across all phases
best_extras = []
best_r2 = r2_base
best_phase = 'baseline'

for src_df, name, getter in [
    (phase_a_df, 'A', lambda r: [r['candidate']]),
    (phase_b_df, 'B', lambda r: [r['cand_1'], r['cand_2']]),
    (phase_c_df, 'C', lambda r: [r['cand_1'], r['cand_2'], r['cand_3']]),
]:
    if len(src_df) == 0:
        continue
    if src_df.iloc[0]['r2'] > best_r2:
        best_r2 = src_df.iloc[0]['r2']
        best_extras = getter(src_df.iloc[0])
        best_phase = name

if not best_extras:
    print('No phase improved over baseline. Stopping here.')
    print(f'Reporting: R2 = {r2_base:.4f} (C6 baseline)')
else:
    print(f'Best: Phase {best_phase}, R2 = {best_r2:.4f}, extras = {best_extras}')
    print(f'\nValidating with seeds 0-9...')
    new_feats = BASELINE + best_extras

    r2s_new, r2s_base_seeds = [], []
    for seed in range(10):
        r2_n, _ = eval_reg_99(new_feats, df_99_ext, y_99, seed=seed)
        r2_b, _ = eval_reg_99(BASELINE, df_99_ext, y_99, seed=seed)
        r2s_new.append(r2_n)
        r2s_base_seeds.append(r2_b)

    print(f'\n10-seed C6 baseline:  R2 = {np.mean(r2s_base_seeds):.4f} +/- {np.std(r2s_base_seeds):.4f}')
    print(f'10-seed with extras:  R2 = {np.mean(r2s_new):.4f} +/- {np.std(r2s_new):.4f}')

    diff = np.mean(r2s_new) - np.mean(r2s_base_seeds)
    pooled_std = np.sqrt(np.std(r2s_new)**2 + np.std(r2s_base_seeds)**2)
    ratio = diff / pooled_std if pooled_std > 0 else 0
    print(f'Difference: {diff:+.4f}, ratio to pooled std: {ratio:.2f}')
    print('  > 2.0 = real signal, 1-2 = marginal, < 1 = noise')

    pd.DataFrame({
        'seed': list(range(10)),
        'r2_baseline': r2s_base_seeds,
        'r2_with_extras': r2s_new,
    }).to_csv(os.path.join(RESULTS_DIR, 'multiseed_validation.csv'), index=False)


## Cell 10: Polymorph check on the best feature set

Specifically: did adding these features change the predictions for the BiITe and STeW polymorphs that the C6 model couldn't distinguish?

In [ ]:
if best_extras:
    new_feats = BASELINE + best_extras
    X_new = df_99_ext[new_feats].fillna(0).values
    X_base = df_99_ext[BASELINE].fillna(0).values

    model = XGBRegressor(**XGB_REG_PARAMS)
    y_pred_new = cross_val_predict(model, X_new, y_99, cv=LeaveOneOut())
    model = XGBRegressor(**XGB_REG_PARAMS)
    y_pred_base = cross_val_predict(model, X_base, y_99, cv=LeaveOneOut())

    polymorph_check_df = df_99[['uid', 'Formula', TARGET]].copy()
    polymorph_check_df['y_pred_C6'] = y_pred_base
    polymorph_check_df['y_pred_new'] = y_pred_new
    polymorph_check_df['resid_C6'] = polymorph_check_df[TARGET] - polymorph_check_df['y_pred_C6']
    polymorph_check_df['resid_new'] = polymorph_check_df[TARGET] - polymorph_check_df['y_pred_new']

    for f in ['BiITe', 'STeW']:
        rows = polymorph_check_df[polymorph_check_df['Formula'] == f]
        if len(rows) > 1:
            print(f'\n{f} polymorphs:')
            print(rows[['uid', TARGET, 'y_pred_C6', 'y_pred_new', 'resid_C6', 'resid_new']].to_string(index=False))

    polymorph_check_df.to_csv(os.path.join(RESULTS_DIR, 'polymorph_check.csv'), index=False)
else:
    print('No best feature set; skipping polymorph check.')
